# Plastic leakage and mobilisation: what we tested and what we found

Two calculations sit here. **Leakage** estimates how many tonnes of plastic reach the environment. **Mobilisation** ranks how easily waste at a site could reach water. Neither uses satellite pixels to measure plastic.

The script outputs are in `3_leakage`.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

pd.set_option("display.max_colwidth", 60, "display.width", 180)
ROOT = Path(r"D:\Guinea_Methodology\plasticwaste_experiments")

from sklearn.metrics import roc_auc_score

sys.path.insert(0, r"C:\Users\ASUS\Guinea_Project\Land Use-Deforestation")
from plasticwaste_tool.leakage_tool import massbalance as MB
from plasticwaste_tool.leakage_tool import parameters as P

OUT = ROOT / "3_leakage" / "outputs"

## 1. The leakage chain

People × waste per person × plastic share gives plastic. Collection, burning and littering then decide how much reaches the environment. This cell runs the platform's own code.

In [2]:
people = json.loads((OUT / "02_mixed_mc.json").read_text())
urban = people["population"] * people["urban_share"]
rural = people["population"] - urban
rows = []
for name in ("waw_2007", "conakry_recent"):
    r = MB.deterministic(urban, rural, name)
    rows.append({"Plastic share": f"{P.PLASTIC_SHARE_SCENARIOS[name].value:.1%}", "Plastic made, t/yr": r["plastic_t_yr"],
                 "To environment, t/yr": r["to_environment_t_yr"], "To water, t/yr": r["to_water_t_yr"]})
pd.DataFrame(rows).round(0)

,Plastic share,"Plastic made, t/yr","To environment, t/yr","To water, t/yr"
0,4.0%,59737.0,18166.0,545.0
1,18.5%,276286.0,84017.0,2521.0


**Reading it.** These are the platform's two scenarios, reproduced. The calculation is the same in both and only the plastic share differs. 4 % is the only national figure, from 2007. 18.5 % is a Conakry figure applied to all of Guinea. So they work as low and high bounds.

## 2. A central estimate

4 % for rural people and 18.5 % for urban people uses each figure where it was measured.

In [3]:
v = pd.read_csv(OUT / "02_variants.csv")
v[["variant", "plastic_kg_cap_yr", "to_env_t_yr", "to_env_kg_cap_yr", "to_water_t_yr"]].round(1)

,variant,plastic_kg_cap_yr,to_env_t_yr,to_env_kg_cap_yr,to_water_t_yr
0,Deployed low: 4 % everywhere,4.5,18165.8,1.4,545.0
1,Deployed high: 18.5 % everywhere,20.9,84016.9,6.4,2520.5
2,"Mixed: 18.5 % urban, 4 % rural",10.9,40307.5,3.1,1209.2
3,"Mixed, no open burning (naive)",10.9,88929.4,6.7,2667.9
4,"Mixed, collected waste dumped openly",10.9,48656.1,3.7,1459.7
5,"High, no open burning (naive)",20.9,185993.7,14.1,5579.8


**Reading it.** The mixed case makes 10.9 kg of plastic per person a year. The published global record gives 0.03 kg a day, or about 11 kg, but that value is rounded, so 9-13 kg is the fair comparison. The 4 % case gives 4.5 kg and the 18.5 % case gives 20.9 kg. Only the mixed case falls inside.

That is a check on plastic made, not on leakage. Leakage is 3.1 kg per person in the mixed case.

Conakry's share rises because its people are almost all urban:

In [4]:
adm = pd.read_csv(OUT / "02_adm1.csv")
LOW, HIGH = 0.04, 0.185
per_u = MB.deterministic(1e6, 0, "waw_2007")["to_environment_t_yr"] / 1e6
per_r = MB.deterministic(0, 1e6, "waw_2007")["to_environment_t_yr"] / 1e6
adm["central"] = adm.pop_urban * per_u * HIGH / LOW + adm.pop_rural * per_r
adm["low"] = adm.pop_urban * per_u + adm.pop_rural * per_r
t = pd.DataFrame({"Region": adm.adm1, "Share of people": adm.pop_share, "Urban share": adm.urban_share,
                  "Share of leakage, one plastic share": adm["low"] / adm["low"].sum(),
                  "Share of leakage, central": adm.central / adm.central.sum()})
t.sort_values("Share of leakage, central", ascending=False).round(3)

,Region,Share of people,Urban share,"Share of leakage, one plastic share","Share of leakage, central"
2,Conakry,0.157,0.997,0.135,0.282
1,Kindia,0.166,0.487,0.162,0.188
0,Kankan,0.203,0.212,0.211,0.156
3,Nzerekore,0.137,0.276,0.140,0.117
4,Boke,0.100,0.228,0.104,0.079
6,Labe,0.087,0.206,0.091,0.066
5,Faranah,0.088,0.198,0.091,0.066
7,Mamou,0.063,0.196,0.066,0.047


## 3. How mobilisation is calculated

For a site, the platform combines:

- **Mass**: 1, 2 or 4 by site size.
- **Wash-off**: rain on the site, slope and land cover. Rain carries waste downhill.
- **Inundation**: the river rises over the site. Estimated from catchment size and distance to river.
- **Reaching a watercourse**: falls with distance to the river.

Rain or flood can move the waste, so the two routes combine as `1 - (1 - washoff) × (1 - inundation)`. The result is a relative rank, not tonnes.

## 4. Why floods?

There are no observations of plastic moving in Guinea, so the score cannot be tested on plastic directly. What can be tested is one link in the chain: **does water reach this place?** Floods are observed from space in the Global Flood Database (Tellman et al., 2021).

The earlier 0.771 used 13 flooded dump points, but they lie in only 2 places, so it was really 2 observations. Here we use 3,000 independent settled places instead, half flooded and half not.

Height above nearest drainage (HAND, Nobre et al., 2011) is how high a place sits above its nearest stream, a standard measure for this.

In [5]:
pd.read_csv(OUT / "01_flood_auc.csv").round(3)

,score,auc,p05,p95
0,Height above nearest drainage (HAND),0.861,0.842,0.879
1,Slope alone,0.818,0.785,0.847
2,JRC flood hazard map (100-year depth),0.797,0.718,0.863
3,Inundation term alone (upstream catchment),0.672,0.530,0.825
4,Mobilisation score (deployed form),0.665,0.505,0.830
5,Distance to river term alone,0.621,0.467,0.780
6,Wash-off term alone,0.437,0.308,0.568


In [6]:
s = pd.read_csv(OUT / "03_sites_improved.csv")
print(json.dumps({k: v for k, v in json.loads((OUT / "03_improved.json").read_text()).items()
                  if k in ("improved_auc", "deployed_auc", "site_spearman_with_deployed", "top15_cells_shared")}, indent=1))
r = s.groupby("region")[["mobilisation_risk", "improved"]].mean().rank(ascending=False).astype(int)
r.columns = ["Rank now", "Rank with HAND"]
r.sort_values("Rank with HAND")

{
 "improved_auc": 0.852903111111111,
 "deployed_auc": 0.6653826666666667,
 "site_spearman_with_deployed": 0.1543757117510281,
 "top15_cells_shared": 3
}


,Rank now,Rank with HAND
region,,
Conakry,5,1
Kankan,2,2
Kindia,3,3
Boke,7,4
Labe,8,5
Nzerekore,1,6
Faranah,4,7
Mamou,6,8


**Reading it.** The current score tells flooded from dry places at 0.67. With HAND as the inundation term it reaches 0.85. This shows the score is better at saying where water reaches. It does not show how much plastic moves, or where it ends up.

The site order changes a lot, so that change needs a decision before it goes on the platform.

## What this means

- The leakage calculation is reproduced, and its range is an honest statement of what is unknown.
- The regional map mostly follows population.
- Mobilisation is a ranking of exposure to water, not a plastic flow.